# 04 · Montos atípicos

**Objetivo:** revisar los procesos cuyo monto adjudicado se aleja mucho del monto esperado.

> **Importante:** un monto atípico **no** prueba una irregularidad. Señala un proceso que
> conviene revisar: puede ser una adjudicación parcial (solo algunos ítems), un error de
> registro o un precio realmente inusual. Verifica cada caso en la fuente oficial (SEACE).

**Requiere:** `contrataciones-ml predict` (genera `data/predictions/pred_monto_esperado.parquet`).

| Tipo de desviación | Regla |
| --- | --- |
| Sobrecosto inusual | z robusto >= umbral **y** desviación >= mínimo (por defecto 3 y +15 %) |
| Subcosto inusual | z robusto <= -umbral **y** desviación <= -mínimo |
| Revisar datos | Razón adjudicado / referencial fuera del rango plausible |
| Normal | Dentro del rango esperado |

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"          # capa gold
FEATURES = ROOT / "data" / "features"        # dataset de modelado
PREDICTIONS = ROOT / "data" / "predictions"  # monto esperado y atípicos
MODELS = ROOT / "models"                     # modelos entrenados
RAW_API = ROOT / "data" / "raw" / "api"      # páginas crudas de la API del OECE
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
COLORES_DESVIACION = {
    "Normal": "#C8CFD8",
    "Sobrecosto inusual": "#B5533C",
    "Subcosto inusual": "#3E6A8A",
    "Revisar datos": "#C9A227",
}
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
porcentaje = mticker.PercentFormatter(1.0)
print("Proyecto:", ROOT)

In [ ]:
from contrataciones.features.dataset import load_gold
from contrataciones.models import report

ruta = PREDICTIONS / "pred_monto_esperado.parquet"
if not ruta.exists():
    raise FileNotFoundError("Ejecuta primero: contrataciones-ml predict")
pred = pd.read_parquet(ruta)
gold = load_gold(CURATED)
detalle = report.describe_processes(pred, gold)
print(f"Modelo {pred['modelo_version'].iloc[0]} | aceptado: {bool(pred['modelo_aceptado'].iloc[0])}")
print(f"Procesos evaluados: {len(pred):,}")

## 1. ¿Cuántos procesos se apartan de lo esperado?

In [ ]:
orden = list(COLORES_DESVIACION)
conteo = pred["tipo_desviacion"].value_counts().reindex(orden, fill_value=0)
plt.bar(conteo.index, conteo.values, color=[COLORES_DESVIACION[t] for t in conteo.index])
plt.title("Procesos por tipo de desviación"); plt.ylabel("N.° de procesos"); plt.show()
pd.DataFrame({"procesos": conteo, "porcentaje": (conteo / len(pred)).map("{:.1%}".format)})

## 2. Monto real frente a monto esperado

Escala logarítmica en ambos ejes: la diagonal es "adjudicado = esperado".

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for tipo in reversed(orden):
    grupo = pred[pred["tipo_desviacion"] == tipo]
    ax.scatter(grupo["monto_esperado_pen"], grupo["monto_adjudicado_pen"], s=10 if tipo == "Normal" else 30,
               color=COLORES_DESVIACION[tipo], label=f"{tipo} ({len(grupo):,})", alpha=0.7)
limites = [pred[["monto_esperado_pen", "monto_adjudicado_pen"]].min().min(),
           pred[["monto_esperado_pen", "monto_adjudicado_pen"]].max().max()]
ax.plot(limites, limites, color=PALETA[6], linewidth=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Monto esperado (S/)"); ax.set_ylabel("Monto adjudicado (S/)")
ax.set_title("Adjudicado frente a esperado"); ax.legend(); plt.show()

## 3. Distribución del z robusto

`z = (residuo - centro) / escala` del segmento. Las líneas marcan el umbral configurado.

In [ ]:
from contrataciones.utils.config_loader import load_config

umbral = load_config("model_config")["anomalies"]["z_threshold"]
ax = pred["z_robusto"].clip(-10, 10).plot.hist(bins=80, title="z robusto (recortado en ±10)", color=PALETA[2])
for signo in (-1, 1):
    ax.axvline(signo * umbral, color=PALETA[4], linestyle="--")
plt.show()
pred["z_robusto"].describe()

## 4. Casos a revisar primero

Ordenados por tipo y por |z|. El reporte Excel (`data/exports/reporte_montos_atipicos.xlsx`)
trae la misma lista.

In [ ]:
casos = detalle[detalle["tipo_desviacion"] != "Normal"].copy()
if casos.empty:
    print("No hay montos atípicos. En la muestra sintética es lo esperado: la razón está acotada "
          "entre 0.75 y 1.10, así que ningún proceso se aleja lo suficiente.")
else:
    casos["abs_z"] = casos["z_robusto"].abs()
    columnas = ["ocid", "entidad", "metodo_detalle", "categoria", "valor_referencial_pen",
                "monto_adjudicado_pen", "monto_esperado_pen", "pct_desviacion", "z_robusto",
                "tipo_desviacion", "es_adjudicado_igual_referencial"]
    display(casos.sort_values(["tipo_desviacion", "abs_z"], ascending=[True, False])[columnas].head(20))

## 5. Montos atípicos y alertas por reglas

¿Los montos atípicos coinciden con las alertas RF01-RF07? Un sobrecosto inusual con nivel de
riesgo alto merece prioridad; uno sin alertas aporta información nueva.

In [ ]:
riesgo = detalle.merge(gold["fact_proceso"][["proceso_key", "nivel_riesgo"]], on="proceso_key", how="left")
pd.crosstab(riesgo["tipo_desviacion"], riesgo["nivel_riesgo"]).reindex(orden, fill_value=0)

## Conclusiones y uso responsable

* Un monto atípico es una señal para priorizar la revisión, no una conclusión.
* Los **subcostos inusuales** suelen ser adjudicaciones parciales o errores de registro; los
  **sobrecostos inusuales** con adjudicado = referencial merecen revisar si el valor referencial
  se publicó antes de la buena pro (ver `01_auditoria_fuga_de_datos.ipynb`).
* El modelo compara contra lo esperado **dado el valor referencial**: no detecta un valor
  referencial inflado.
* Si el modelo no fue aceptado, interpreta los resultados con cautela.
* Estos resultados alimentan la página **Montos atípicos** del dashboard de Power BI.